# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Access metadata - do not subscript the metadata object, use attributes
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Dataset Description: {metadata.description}")
print(f"Published: {getattr(metadata, 'date_published', getattr(metadata, 'datePublished', 'N/A'))}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list the record sets along with their fields as defined by their `@id` and show field details. All fields and columns are referenced directly by `@id` for robust reproducibility.

In [ ]:
# List all record sets in the dataset, show their `@id` and their fields' `@id`
print("Available record sets:")
record_sets = dataset.metadata.record_sets  # List of RecordSet objects
for i, record_set in enumerate(record_sets):
    print(f"[{i}] RecordSet @id: {record_set.id}")
    field_ids = [(fld.id, fld.name) for fld in getattr(record_set, 'fields', [])]
    print("    Fields:")
    for fid, fname in field_ids:
        print(f"        Field @id: {fid:50} | Name: {fname}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We will load **all** record sets into Pandas DataFrames by their `@id`. You may select a record set to analyze depending on your exploration goal.

In [ ]:
# Prepare DataFrames for each record set
dataframes = {}
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("Loading data for each RecordSet (by @id):")
for rsid in record_set_ids:
    print(f"- Loading: {rsid}")
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)
print("\nLoaded DataFrames:")
for rsid, df in dataframes.items():
    print(f"{rsid}: {df.shape[0]} rows, {df.shape[1]} columns")
# Display the columns for the first record set (choose an appropriate one for analysis)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns of main RecordSet {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Explore and process the data from your chosen record set. Use only field and column names or `@id` as shown above for reproducibility. 

Common EDA includes filtering records based on selected criteria, normalizing numeric fields, detecting or handling outliers, or grouping by key attributes.

Below, we select a numeric field (such as "MW [Da]" for molecular weight), filter by a threshold, normalize values, and group by another key attribute, all using their `@id`.

In [ ]:
# Example: Select and analyze a numeric field - here, Molecular Weight ('MW [Da]')
# Assign variables by field/column @id for reproducibility

# You can check available columns by inspecting: print(dataframes[main_record_set_id].columns)
# For this dataset, typical fields are: Accession, Description, MW [Da], pI, etc.
# Let's try 'MW [Da]' as a numeric field

# Replace 'MW [Da]' with column name or field @id as required
df = dataframes[main_record_set_id]

# Identify a numeric field (adjust the id if different in your data)
numeric_field = 'MW [Da]'
if numeric_field in df.columns:
    try:
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    except Exception as e:
        print(f'Could not convert {numeric_field} to numeric: {e}')
else:
    print(f"Column '{numeric_field}' not in main record set. Available columns: {df.columns.tolist()}")

threshold = 10000  # e.g., molecular weights above 10,000 Da
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field (@id or column name), e.g., 'Description' if present and not too unique
group_field = 'Description'
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (showing means):")
    display(grouped_df.head())
else:
    print(f"Group field '{group_field}' not present in filtered data.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of Molecular Weight
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field} in Main Record Set")
plt.show()

# If another numeric field exists, plot relationship
possible_corr_fields = [col for col in df.columns if col not in [numeric_field] and pd.api.types.is_numeric_dtype(df[col])]
if possible_corr_fields:
    corr_field = possible_corr_fields[0]
    plt.figure(figsize=(6,4))
    sns.scatterplot(x=numeric_field, y=corr_field, data=df)
    plt.xlabel(numeric_field)
    plt.ylabel(corr_field)
    plt.title(f"Scatter: {numeric_field} vs {corr_field}")
    plt.show()
else:
    print("No other numeric field for correlation plot.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the mass spectrometry dataset using its Croissant schema, listed record sets and their properties (`@id`), extracted data by record set, analyzed numeric fields (e.g., molecular weight), and visualized distributions. All data access is reproducible via field and record set `@id`s as recommended by the Croissant specification.

You may adapt and extend these analyses for further hypothesis testing, machine learning tasks, or deeper biological insight as needed.